# Step 04: Transformer block and GPT

Tests for `04-transformer/gpt.py`. Write code in the `.py` file, only import and test here.

Step 03 had one attention layer. A real transformer adds an MLP, residual connections, LayerNorm, and stacks many blocks.

Run the cells top to bottom (Run All).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

sys.path.append("04-transformer")
from gpt import (
    VOCAB_SIZE, get_device, load_data, split_data, decode, get_batch, train,
    FeedForward, Block, GPT,
)

device = get_device()
print("device:", device)

## 1. FeedForward (the MLP)

Attention moves information *between* tokens. The MLP then processes each token *on its own*: expand to 4x wider, nonlinearity (GELU), shrink back.

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 6, 16)          # (B=2, T=6, C=16)
ffn = FeedForward(n_embd=16)

print(ffn(x).shape)                # expect (2, 6, 16): same shape in and out
print(sum(p.numel() for p in ffn.parameters()))   # 16*64+64 + 64*16+16 = 2128

## 2. LayerNorm

Rescales each token's vector to mean 0 and standard deviation 1, so values stay at a stable size as they pass through many layers.

In [ ]:
x = torch.randn(2, 6, 16) * 3 + 5   # deliberately off-scale: mean ~5, std ~3
print(x.mean().item(), x.std().item())

ln = nn.LayerNorm(16)
y = ln(x)
print(y.mean(dim=-1)[0])           # expect ~0 for every token
print(y.std(dim=-1)[0])            # expect ~1 for every token

## 3. One transformer block

`x = x + attn(ln(x))` then `x = x + ffn(ln(x))`. The `x +` is a residual connection: each sub-layer only learns a *correction* to x. At the start (small random weights) the block is close to doing nothing, which makes deep stacks trainable.

In [ ]:
torch.manual_seed(0)
block = Block(n_embd=16, n_heads=4, block_size=8)
block.apply(GPT._init_weights)     # same small init the full GPT uses

x = torch.randn(2, 6, 16)
y = block(x)
print(y.shape)                     # expect (2, 6, 16)
print("size of x:           ", x.abs().mean().item())
print("size of what it adds:", (y - x).abs().mean().item())   # expect much smaller than x

## 4. The full GPT

Token + position embeddings -> N blocks -> final LayerNorm -> output layer.

In [ ]:
data = load_data("02-bigram/data/input.txt")
train_data, val_data = split_data(data)

torch.manual_seed(0)
model = GPT(VOCAB_SIZE, n_embd=128, n_heads=4, n_layers=4, block_size=64).to(device)
print(f"parameters: {model.num_params():,}")
print(len(model.blocks), "blocks")

xb, yb = get_batch(train_data, block_size=64, batch_size=8, device=device)
logits, loss = model(xb, yb)
print(logits.shape)                # expect (8, 64, 256)
print(loss.item())                 # expect ~5.55 = ln(256): small init means a near-uniform guess

## 5. Where do the parameters live?

Each block costs about `12 * C^2` parameters (attention `4C^2`, MLP `8C^2`). Stacking blocks is where the size comes from.

In [ ]:
C = 128
for name in ["tok_emb", "pos_emb", "blocks", "ln_f", "lm_head"]:
    n = sum(p.numel() for p in getattr(model, name).parameters())
    print(f"{name:8s} {n:>9,}")
per_block = sum(p.numel() for p in model.blocks[0].parameters())
print(f"\none block: {per_block:,}   vs 12*C^2 = {12 * C * C:,}")

## 6. Train

Same training loop as steps 02 and 03, so any gain comes from the architecture. Takes about a minute.

In [ ]:
torch.manual_seed(0)
model = GPT(VOCAB_SIZE, n_embd=128, n_heads=4, n_layers=4, block_size=64).to(device)
losses = train(model, train_data, steps=3000, batch_size=64, block_size=64,
               lr=2e-3, device=device)

final = sum(losses[-100:]) / 100
print(f"first 100 steps avg: {sum(losses[:100]) / 100:.3f}")
print(f"last 100 steps avg:  {final:.3f}")   # expect about 1.4 (bigram 2.47, one attention layer 2.05)

plt.plot(losses, alpha=0.3)
plt.axhline(2.47, color="red", linestyle="--", label="bigram (step 02)")
plt.axhline(2.05, color="orange", linestyle="--", label="one attention layer (step 03)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("GPT training loss")
plt.show()

## 7. Validation loss

Watch the gap between train and val. A bigger model starts to memorize its training text.

In [ ]:
@torch.no_grad()
def eval_loss(model, data, n_batches=50):
    total = 0.0
    for _ in range(n_batches):
        xb, yb = get_batch(data, block_size=64, batch_size=64, device=device)
        _, loss = model(xb, yb)
        total += loss.item()
    return total / n_batches

print("train:", round(eval_loss(model, train_data), 3))
print("val:  ", round(eval_loss(model, val_data), 3))   # expect train ~1.4, val ~1.65: the gap means it is starting to memorize

## 8. Generate

Compare with step 03: more real words, line structure, speaker names followed by colons, short coherent phrases.

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = model.generate(start, max_new_tokens=400)
print(decode(out[0].tolist()))

## 9. Weight tying

The input embedding (token -> vector) and the output layer (vector -> token scores) have the same shape, so they can share one matrix. GPT-2 does this. It saves `vocab * C` parameters, which matters once the vocabulary is 50k instead of 256 (step 05).

In [ ]:
plain = GPT(VOCAB_SIZE, n_embd=128, n_heads=4, n_layers=4, block_size=64)
tied = GPT(VOCAB_SIZE, n_embd=128, n_heads=4, n_layers=4, block_size=64, tie_weights=True)

print(plain.num_params(), tied.num_params())          # tied is smaller
print(plain.num_params() - tied.num_params(), "=", VOCAB_SIZE * 128)   # exactly vocab * C
print(tied.lm_head.weight is tied.tok_emb.weight)    # True: one shared matrix

## What is next

The model is complete, but we trained it for 3000 steps with a fixed learning rate on 1MB of text. Step 05 trains properly: a bigger dataset (TinyStories), a real tokenizer, learning-rate warmup and decay, gradient clipping, checkpoints.